# FIFA 23 Player Data — Preprocessing

This notebook loads the raw FIFA 23 player dataset, selects the relevant columns,
cleans and transforms the data, and exports `cleaned_players.csv` ready for the Dash dashboard.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

DATA_DIR = Path('../data')

## 1. Load, Inspect & Column Selection

Read the raw CSV using chunked loading to stay within memory limits, keep only the 22 columns needed to power all 9 required chart types, and filter to FIFA 23 only.

| Column | Used For |
|--------|----------|
| `short_name`, `long_name` | Player labels / tooltips |
| `player_positions` | Position group engineering |
| `overall` | All comparison and distribution charts |
| `potential` | Violin chart; bubble size |
| `value_eur` → `market_value_in_eur` | Bubble chart Y-axis |
| `wage_eur` → `wage_in_eur` | Box plot Y-axis |
| `age` | Scatter X-axis; area/line X-axis |
| `height_cm`, `weight_kg` | Supporting physical attributes |
| `league_name` | Stacked / clustered chart grouping |
| `club_name` | Column / bar chart categories |
| `nationality_name` | Bar chart categories |
| `preferred_foot` | Violin chart X-axis |
| `work_rate` | Alternative violin grouping |
| `pace`, `shooting`, `passing`, `dribbling`, `defending`, `physic` | Skill breakdown for stacked/clustered |
| `fifa_version` | Time-series X-axis (line & area charts) |

In [15]:
KEEP_COLS = [
    'short_name', 'long_name', 'player_positions',
    'overall', 'potential',
    'value_eur', 'wage_eur',
    'age', 'height_cm', 'weight_kg',
    'league_name', 'club_name', 'nationality_name',
    'preferred_foot', 'work_rate',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'fifa_version',
]

chunks = []
for chunk in pd.read_csv(DATA_DIR / 'male_players.csv', usecols=KEEP_COLS,
                          chunksize=50000, low_memory=False, on_bad_lines='skip'):
    chunks.append(chunk[chunk['fifa_version'] == 23])

df = pd.concat(chunks, ignore_index=True)

df = df.rename(columns={
    'value_eur': 'market_value_in_eur',
    'wage_eur':  'wage_in_eur',
})

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Shape: (166674, 22)
Columns: ['fifa_version', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'market_value_in_eur', 'wage_in_eur', 'age', 'height_cm', 'weight_kg', 'league_name', 'club_name', 'nationality_name', 'preferred_foot', 'work_rate', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']


,fifa_version,short_name,long_name,player_positions,overall,potential,market_value_in_eur,wage_in_eur,age,height_cm,...,club_name,nationality_name,preferred_foot,work_rate,pace,shooting,passing,dribbling,defending,physic
0,23,L. Messi,Lionel Andrés Messi Cuccittini,RW,91,91,54000000.0,195000.0,35,169,...,Paris Saint Germain,Argentina,Left,Low/Low,81.0,89.0,90.0,94.0,34.0,64.0
1,23,K. Benzema,Karim Benzema,"CF, ST",91,91,64000000.0,450000.0,34,185,...,Real Madrid,France,Right,Medium/Medium,80.0,88.0,83.0,87.0,39.0,78.0
2,23,R. Lewandowski,Robert Lewandowski,ST,91,91,84000000.0,420000.0,33,185,...,FC Barcelona,Poland,Right,High/Medium,75.0,91.0,79.0,86.0,44.0,83.0
3,23,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,107500000.0,350000.0,31,181,...,Manchester City,Belgium,Right,High/Medium,74.0,88.0,93.0,87.0,63.0,77.0
4,23,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,95,190500000.0,230000.0,23,182,...,Paris Saint Germain,France,Right,High/Low,97.0,89.0,80.0,92.0,36.0,76.0


## 2. Drop Missing Values

Rows that are missing any critical column are dropped because those charts cannot render without them.
Non-critical columns (e.g. `height_cm`) are left as-is — charts that use them will handle NaN locally.

In [16]:
CRITICAL = [
    'overall', 'age', 'market_value_in_eur', 'wage_in_eur',
    'club_name', 'nationality_name', 'player_positions',
]

before = len(df)
df = df.dropna(subset=CRITICAL)
after = len(df)

print(f'Dropped : {before - after:,} rows with missing critical values')
print(f'Remaining: {after:,} rows')
df.isnull().sum()

Dropped : 820 rows with missing critical values
Remaining: 165,854 rows


fifa_version               0
short_name                 0
long_name                  0
player_positions           0
overall                    0
potential                  0
market_value_in_eur        0
wage_in_eur                0
age                        0
height_cm                  0
weight_kg                  0
league_name                0
club_name                  0
nationality_name           0
preferred_foot             0
work_rate                  0
pace                   18454
shooting               18454
passing                18454
dribbling              18454
defending              18454
physic                 18454
dtype: int64

## 3. Position Group Engineering

We collapse the many specific position codes into four broad groups used as chart categories:

- **GK** — Goalkeeper
- **DEF** — Defenders (CB, LB, RB, LWB, RWB)
- **MID** — Midfielders (CDM, CM, CAM, LM, RM)
- **FWD** — Forwards (ST, CF, LW, RW, etc.)

We use only the *first listed position* for each player.

In [17]:
DEFENDERS  = {'CB', 'LB', 'RB', 'LWB', 'RWB', 'SW'}
MIDFIELDERS = {'CDM', 'CM', 'CAM', 'LM', 'RM', 'DM', 'LAM', 'RAM', 'RCM', 'LCM', 'RDM', 'LDM'}

def assign_position_group(positions):
    primary = str(positions).split(',')[0].strip().upper()
    if primary == 'GK':
        return 'GK'
    elif primary in DEFENDERS:
        return 'DEF'
    elif primary in MIDFIELDERS:
        return 'MID'
    else:
        return 'FWD'

df['position_group'] = df['player_positions'].apply(assign_position_group)

print('Position group distribution:')
print(df['position_group'].value_counts())

Position group distribution:
position_group
MID    59593
DEF    55742
FWD    32065
GK     18454
Name: count, dtype: int64


## 4. Outlier Capping

A handful of elite players (e.g. Mbappe, Messi) have market values an order of magnitude above everyone else.
Capping at the 99th percentile keeps the bubble chart readable without removing those players entirely.

In [18]:
cap_value = df['market_value_in_eur'].quantile(0.99)
df['market_value_in_eur'] = df['market_value_in_eur'].clip(upper=cap_value)

print(f'99th percentile cap applied: €{cap_value:,.0f}')
print()
print(df['market_value_in_eur'].describe().apply(lambda x: f'{x:,.2f}'))

99th percentile cap applied: €36,500,000

count       165,854.00
mean      2,609,762.42
std       5,489,104.24
min           9,000.00
25%         475,000.00
50%       1,000,000.00
75%       2,000,000.00
max      36,500,000.00
Name: market_value_in_eur, dtype: object


## 5. Export Cleaned Dataset

Save the final cleaned dataframe to `data/cleaned_players.csv`.

In [19]:
output_path = DATA_DIR / 'cleaned_players.csv'
df.to_csv(output_path, index=False)

print(f'Saved to : {output_path.resolve()}')
print(f'Final shape: {df.shape}')
print(f'Columns   : {df.columns.tolist()}')
df.head()

Saved to : C:\Users\itski\Desktop\Fifa 23 Players\Team_FIFA23_Analysis\data\cleaned_players.csv
Final shape: (165854, 23)
Columns   : ['fifa_version', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'market_value_in_eur', 'wage_in_eur', 'age', 'height_cm', 'weight_kg', 'league_name', 'club_name', 'nationality_name', 'preferred_foot', 'work_rate', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'position_group']


,fifa_version,short_name,long_name,player_positions,overall,potential,market_value_in_eur,wage_in_eur,age,height_cm,...,nationality_name,preferred_foot,work_rate,pace,shooting,passing,dribbling,defending,physic,position_group
0,23,L. Messi,Lionel Andrés Messi Cuccittini,RW,91,91,36500000.0,195000.0,35,169,...,Argentina,Left,Low/Low,81.0,89.0,90.0,94.0,34.0,64.0,FWD
1,23,K. Benzema,Karim Benzema,"CF, ST",91,91,36500000.0,450000.0,34,185,...,France,Right,Medium/Medium,80.0,88.0,83.0,87.0,39.0,78.0,FWD
2,23,R. Lewandowski,Robert Lewandowski,ST,91,91,36500000.0,420000.0,33,185,...,Poland,Right,High/Medium,75.0,91.0,79.0,86.0,44.0,83.0,FWD
3,23,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,36500000.0,350000.0,31,181,...,Belgium,Right,High/Medium,74.0,88.0,93.0,87.0,63.0,77.0,MID
4,23,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,95,36500000.0,230000.0,23,182,...,France,Right,High/Low,97.0,89.0,80.0,92.0,36.0,76.0,FWD
